# Cleanup: Glossary Terms

This notebook cleans up resources created by `setup/03_apply_glossary_terms.ipynb`.

**Resources that will be deleted:**

**1. Dataplex Glossary Resources:**
   - Business Glossary: `acs-demographics-glossary`
   - All glossary categories (5 categories)
   - All glossary terms (up to 96 terms)
   - Entry links between glossary terms and BigQuery columns

**⚠️  WARNING:**
- This operation cannot be undone
- All glossary resources will be permanently deleted
- Entry links to BigQuery columns will be removed

**Required permissions:**
- `roles/dataplex.catalogAdmin` (to delete glossary and its resources)

## Section 1: Configuration & Authentication

In [ ]:
# Configuration variables
PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           
CATALOG_LOCATION = "us"  # BigQuery US multi-region datasets are cataloged in 'us'

# Dataplex Glossary
GLOSSARY_ID = "acs-demographics-glossary"

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Catalog Location: {CATALOG_LOCATION}")
print(f"  Glossary to delete: {GLOSSARY_ID}")

In [ ]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

In [ ]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = ["requests"]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")

In [ ]:
# Initialize clients
import requests
from google.auth.transport.requests import Request

# Get credentials for REST API calls (needed for glossary operations)
credentials.refresh(Request())

print("✅ Clients initialized:")
print("  - Authentication credentials")

## Section 2: List Glossary Terms

First, we'll list all terms in the glossary to see what needs to be deleted.

In [ ]:
# List all terms in the glossary
print("📋 Listing glossary terms...")
print()

glossary_path = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/glossaries/{GLOSSARY_ID}"
terms_url = f"https://dataplex.googleapis.com/v1/{glossary_path}/terms"

headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

try:
    response = requests.get(terms_url, headers=headers)
    
    if response.status_code == 200:
        terms_data = response.json()
        terms = terms_data.get('glossaryTerms', [])
        print(f"✅ Found {len(terms)} glossary terms")
        print()
    elif response.status_code == 404:
        print(f"⚠️  Glossary '{GLOSSARY_ID}' not found - skipping glossary cleanup")
        terms = []
    else:
        print(f"⚠️  Could not list terms (status {response.status_code})")
        print(f"   Response: {response.text[:200]}")
        terms = []
        
except Exception as e:
    print(f"⚠️  Error listing terms: {e}")
    terms = []

## Section 3: Delete Glossary Terms

Delete all glossary terms. This will also remove entry links to BigQuery columns.

In [ ]:
# Delete all glossary terms
print("🗑️  Deleting glossary terms...")
print()

if len(terms) == 0:
    print("⚠️  No terms to delete")
    deleted_terms = 0
    error_terms = 0
else:
    deleted_terms = 0
    error_terms = 0
    
    for term in terms:
        term_name = term.get('name', '')
        term_id = term_name.split('/')[-1] if term_name else 'unknown'
        
        try:
            delete_response = requests.delete(
                f"https://dataplex.googleapis.com/v1/{term_name}",
                headers=headers
            )
            
            if delete_response.status_code in [200, 204]:
                deleted_terms += 1
            elif delete_response.status_code == 404:
                deleted_terms += 1  # Already deleted
            else:
                error_terms += 1
                print(f"   ⚠️  Failed to delete term {term_id}: {delete_response.status_code}")
        
        except Exception as e:
            error_terms += 1
            print(f"   ⚠️  Error deleting term {term_id}: {str(e)[:80]}")
    
    print()
    print(f"✅ Glossary terms deletion completed:")
    print(f"   Deleted: {deleted_terms}")
    print(f"   Errors: {error_terms}")

## Section 4: List and Delete Categories

Delete all glossary categories.

In [ ]:
# List and delete all categories
print()
print("📋 Listing glossary categories...")
print()

categories_url = f"https://dataplex.googleapis.com/v1/{glossary_path}/categories"

try:
    response = requests.get(categories_url, headers=headers)
    
    if response.status_code == 200:
        categories_data = response.json()
        categories = categories_data.get('glossaryCategories', [])
        print(f"✅ Found {len(categories)} categories")
        print()
    elif response.status_code == 404:
        print(f"⚠️  Glossary not found - skipping category deletion")
        categories = []
    else:
        print(f"⚠️  Could not list categories (status {response.status_code})")
        categories = []
        
except Exception as e:
    print(f"⚠️  Error listing categories: {e}")
    categories = []

# Delete categories
print("🗑️  Deleting categories...")
print()

if len(categories) == 0:
    print("⚠️  No categories to delete")
    deleted_categories = 0
    error_categories = 0
else:
    deleted_categories = 0
    error_categories = 0
    
    for category in categories:
        category_name = category.get('name', '')
        category_id = category_name.split('/')[-1] if category_name else 'unknown'
        
        print(f"   Deleting: {category_id}")
        
        try:
            delete_response = requests.delete(
                f"https://dataplex.googleapis.com/v1/{category_name}",
                headers=headers
            )
            
            if delete_response.status_code in [200, 204]:
                print(f"   ✅ Deleted: {category_id}")
                deleted_categories += 1
            elif delete_response.status_code == 404:
                print(f"   ⚠️  Not found: {category_id} (already deleted)")
                deleted_categories += 1
            else:
                print(f"   ❌ Failed: {category_id} - Status {delete_response.status_code}")
                error_categories += 1
        
        except Exception as e:
            print(f"   ❌ Error: {category_id} - {str(e)[:60]}")
            error_categories += 1
    
    print()
    print(f"✅ Categories deletion completed:")
    print(f"   Deleted: {deleted_categories}")
    print(f"   Errors: {error_categories}")

## Section 5: Delete Glossary

Finally, delete the glossary itself.

In [ ]:
# Delete the glossary itself
print()
print("🗑️  Deleting glossary...")
print()

glossary_delete_url = f"https://dataplex.googleapis.com/v1/{glossary_path}"

try:
    delete_response = requests.delete(glossary_delete_url, headers=headers)
    
    if delete_response.status_code in [200, 204]:
        print(f"✅ Glossary '{GLOSSARY_ID}' deleted successfully")
    elif delete_response.status_code == 404:
        print(f"⚠️  Glossary '{GLOSSARY_ID}' not found (already deleted)")
    else:
        print(f"❌ Failed to delete glossary (status {delete_response.status_code})")
        print(f"   Response: {delete_response.text[:200]}")
        
except Exception as e:
    print(f"❌ Error deleting glossary: {e}")

print()
print("=" * 60)
print("✅ Glossary cleanup completed!")
print("=" * 60)

## Section 6: Cleanup Summary

In [ ]:
# Summary
print()
print("=" * 70)
print("🎉 CLEANUP COMPLETED!")
print("=" * 70)
print()
print("📊 Summary of deleted resources:")
print()
print("   1. Dataplex Glossary:")
print(f"      - Glossary: {GLOSSARY_ID}")
print(f"      - Categories: {deleted_categories if 'deleted_categories' in locals() else 0}")
print(f"      - Terms: {deleted_terms if 'deleted_terms' in locals() else 0}")
print()
print("✅ All resources from apply_glossary_terms.ipynb have been cleaned up!")
print()
print("🔗 Verify in Console:")
print(f"   Dataplex Glossaries: https://console.cloud.google.com/dataplex/dp-glossaries?project={PROJECT_ID}")